# ROCLING 2026 DSA — L2 → L3 → L1 完整 pipeline (Colab GPU)

對新住民自我反思文本預測 valence / arousal (1-9)。本 notebook 一路從 **L2** 跑到 **L3** 再到 **L1 訓練**：

| 階段 | 程式 | 產出 | 說明 |
|---|---|---|---|
| **L2** | `word_va_regressor.py` | `outputs/l2_word_va.pkl` | FastText + SVR 學「詞→VA」，覆蓋率無限擴充 |
| **L3** | `affective_graph.py` | `outputs/l3_graph.pkl` | 情感知識圖譜（節點=情緒詞VA，邊=FastText kNN）|
| **L1** | `train.py` | `outputs/best_model.pt`、`outputs/submission.csv` | MacBERT + 詞典特徵融合，雙回歸頭 |

**重點**：上傳的 zip **不含** 814MB 的 `l2_word_va.pkl`——Colab 重建後再下載回本機（見各下載 cell），確保之後可復現。

**使用前**：上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**，然後依序執行每個 cell。

In [ ]:
# 1) 安裝套件
# gensim>=4.3.3 修掉新版 scipy 移除 triu 的相容問題；其餘 Colab 多半已內建
!pip -q install "transformers>=4.40" "gensim>=4.3.3" jieba networkx scikit-learn scipy
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2) 上傳 baseline.zip（不含 outputs/）並解壓、切目錄
from google.colab import files
up = files.upload()              # 選 baseline_l3.zip（或任何含 train.py 的 zip）
import zipfile, os
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z: z.extractall('.')
for root, _, fs in os.walk('.'):
    if 'train.py' in fs and 'data' in os.listdir(root):
        os.chdir(root); break
os.makedirs('outputs', exist_ok=True)
print('工作目錄:', os.getcwd()); print(sorted(os.listdir('.')))

## L2 — 重建「詞→VA」迴歸器

FastText (char n-gram) 在 train.csv 上學詞向量，SVR 學「詞向量→VA」。產出 `outputs/l2_word_va.pkl`（含 `fasttext`、`svr_v`、`svr_a` 三個物件）。約幾分鐘。

In [ ]:
# 3) 重建 L2（產生 outputs/l2_word_va.pkl）
!python word_va_regressor.py

In [ ]:
# 4) 下載 l2_word_va.pkl 回本機（約 800MB，存到 baseline/outputs/ 即可復現 L3）
from google.colab import files
print('size:', round(os.path.getsize('outputs/l2_word_va.pkl')/1e6, 1), 'MB')
files.download('outputs/l2_word_va.pkl')

## L3 — 建情感知識圖譜

用 L2 的 FastText 向量做 kNN 連邊、CVAW/CVAP 的 VA 當節點屬性，建成 networkx 圖。產出 `outputs/l3_graph.pkl`，並印出統計、語義鄰居、高 arousal 種子詞、可控生成 prompt 範例。

In [ ]:
# 5) 建圖 + 存檔 + 看 demo
import pickle
from affective_graph import build_graph, neighbors, seeds_for_target, build_generation_prompt
G, terms = build_graph(k=8)
with open('outputs/l3_graph.pkl', 'wb') as f:
    pickle.dump(G, f)
print(f'節點={G.number_of_nodes()}  邊={G.number_of_edges()}  平均度={2*G.number_of_edges()/G.number_of_nodes():.1f}')
for w in ['焦慮', '開心']:
    nb = neighbors(G, w)
    if nb: print(f'「{w}」鄰居:', '  '.join(f'{x[0]}(A={x[2]:.1f})' for x in nb))
seeds = seeds_for_target(G, a_range=(7.0, 9.0), n=12)
print('\n高 arousal 種子詞:', '、'.join(w for w, _, _ in seeds))

In [ ]:
# 6) 下載 l3_graph.pkl 回本機
from google.colab import files
files.download('outputs/l3_graph.pkl')

## L1 — 訓練 MacBERT + 詞典特徵融合（實驗 4，目前最佳）

詞典融合預設開啟（`external/emobank` 的 CVAW/CVAP）。產出 `outputs/best_model.pt` + `outputs/submission.csv`。

> 復現其他實驗：消融詞典加 `--no_lexicon`；DAPT 版先跑下一個 markdown 區的 `dapt.py` 再 `--model outputs/dapt_macbert`。

In [ ]:
# 7) 訓練 L1（實驗 4）
!python train.py --epochs 4 --batch_size 32 --model hfl/chinese-macbert-base

In [ ]:
# 8) 檢視 submission
import pandas as pd
df = pd.read_csv('outputs/submission.csv')
print(df.describe()); df.head(10)

In [ ]:
# 9) 下載訓練好的模型 + submission 回本機（best_model.pt 約數百 MB）
from google.colab import files
files.download('outputs/submission.csv')
files.download('outputs/best_model.pt')

## （可選）復現實驗 1–3

原始實驗程式碼都保留著，可直接復現：
- **實驗 1 Baseline**：`!python train.py --epochs 4 --batch_size 32 --no_lexicon`（用 `data/orign_train_data.csv` 需先 `prepare_data.py`）
- **實驗 2 DAPT**：`!python dapt.py --epochs 30 --batch_size 16` 再 `!python train.py --model outputs/dapt_macbert`
- **實驗 3 反思語料**：`!python train.py --no_lexicon`（用合併後的 `data/train.csv`）

資料集重建：`!python prepare_data.py`（合併 EmoBank + DSA-MST + ROCLING-2021）。